# 8: Experiment 3 - Necessity Test (Silencing)

## Test 1: Minimal Validation (2 neurons × 2 freqs × 10T)

**Purpose**
- Validate Exp3 logic (control + silencing)
- Observe memory accumulation pattern (10 trials)
- Test silencing mechanism works correctly

**Configuration**
- **Neurons to silence**: 2 neurons (21-22, skip first 20 GRNs)
- **Frequencies**: [50, 100] Hz
- **Trials**: 10 (match full-scale target)
- **Workers**: 1 (serial, for clean observation)
- **Expected runtime**: 2-3 minutes

**Memory Watch**
- Each simulation may leave ~15MB residue
- 2 neurons × 2 freqs × (1 control) = 5 simulations
- Expected accumulation: ~75MB (negligible)

---

## 8.1

In [17]:
# ==================== Cell 8.2: Exp3测试1配置 ====================

from flylif.utils.memory_utils import MemoryMonitor, print_memory
import gc

print("=" * 70)
print("Exp3 Test 1: Minimal Validation (2×2×10T, Worker Mode)")
print("=" * 70)

# Configuration
NEU_ACTIVATE = NEU_SUGAR_LEFT
NEURONS_SILENCE_TEST = top_200_neurons[20:22]   # 2 neurons (skip GRNs)
FREQS_TEST1 = [50, 100]
N_TRIALS_TEST1 = 10
TARGET_MN9_TEST = NEU_MN9_RIGHT

print(f"\nConfiguration:")
print(f"  Activated: {len(NEU_ACTIVATE)} Sugar GRNs")
print(f"  To silence: {len(NEURONS_SILENCE_TEST)} neurons (IDs: {list(NEURONS_SILENCE_TEST)})")
print(f"  Frequencies: {FREQS_TEST1}")
print(f"  Trials: {N_TRIALS_TEST1}")
print(f"  Total tasks: {len(FREQS_TEST1) + len(NEURONS_SILENCE_TEST)*len(FREQS_TEST1)}")
print(f"    - Control: {len(FREQS_TEST1)}")
print(f"    - Silencing: {len(NEURONS_SILENCE_TEST) * len(FREQS_TEST1)}")
print(f"\n  Method: Parallel workers (n_jobs=2, avoid start_scope conflict)")

print_memory("\nInitial memory: ")

Exp3 Test 1: Minimal Validation (2×2×10T, Worker Mode)

Configuration:
  Activated: 21 Sugar GRNs
  To silence: 2 neurons (IDs: [720575940624963786, 720575940622695448])
  Frequencies: [50, 100]
  Trials: 10
  Total tasks: 6
    - Control: 2
    - Silencing: 4

  Method: Parallel workers (n_jobs=2, avoid start_scope conflict)

Initial memory: Memory: 45.1% used, 9.4GB free, Swap: 0.0GB


{'used_gb': 7.704223744,
 'available_gb': 9.43104,
 'percent': 45.1,
 'swap_gb': 0.0}

In [78]:
# ==================== Cell 8.3A: Exp3 Worker函数定义 ====================

def exp3_single_condition_worker(condition_type, neuron_id, freq, neu_exc, 
                                 data, params, target_neurons, n_trials):
    """
    Single condition worker (control or silencing).
    
    Each worker runs in independent process to avoid start_scope conflicts.
    
    Parameters
    ----------
    condition_type : str
        'control' or 'silencing'
    neuron_id : int or None
        Neuron to silence (None for control)
    freq : int
        Frequency (Hz)
    neu_exc : list
        Neurons to activate
    ...
    
    Returns
    -------
    tuple : (condition_type, freq, neuron_id, stats)
    """
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    import gc
    
    # Build network (in independent process)
    columns = data['columns']
    NET = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    # Run simulation
    if condition_type == 'control':
        result = run_simulation(
            net_components=NET,
            neu_exc=neu_exc,
            params={'r_poi': freq * Hz},
            n_trials=n_trials,
            verbose=False
        )
    else:  # silencing
        result = run_simulation(
            net_components=NET,
            neu_exc=neu_exc,
            neu_slnc=[neuron_id],
            params={'r_poi': freq * Hz},
            n_trials=n_trials,
            verbose=False
        )
    
    df = result['df']
    duration_s = float(params['t_run'] / ms) / 1000
    
    # Calculate target rates per trial
    target_rates = []
    for trial in range(n_trials):
        trial_df = df[df['trial'] == trial]
        count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                   for tid in target_neurons)
        target_rates.append(count / duration_s)
    
    stats = {
        'mean': np.mean(target_rates),
        'std': np.std(target_rates),
    }
    
    # Cleanup
    del NET
    del result
    gc.collect()
    
    return (condition_type, freq, neuron_id, stats)

print("✅ Worker function defined")

✅ Worker function defined


### debug

In [80]:
# ==================== Cell 8.3_debug: 测试worker基本功能 ====================

from joblib import Parallel, delayed

def simple_worker_test(task_id, data):
    """最简单的worker，只build网络"""
    from flylif.core.network import build_network
    from time import time
    
    print(f"  Worker {task_id} started")
    t0 = time()
    
    columns = data['columns']
    NET = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=DEFAULT_PARAMS,
        syn_threshold=5,
        verbose=False
    )
    
    build_time = time() - t0
    n_neurons = NET['n_neurons']
    
    del NET
    
    return (task_id, build_time, n_neurons)

print("\n测试：2个worker各build 1次网络...")
t0 = time()

results = Parallel(n_jobs=2, verbose=10)(
    delayed(simple_worker_test)(i, DATA) for i in range(2)
)

print(f"\n✅ 完成: {time()-t0:.1f}s")
for task_id, build_time, n_neurons in results:
    print(f"  Worker {task_id}: build {build_time:.1f}s, {n_neurons:,} neurons")


测试：2个worker各build 1次网络...


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


  Worker 1 started
  Worker 0 started

✅ 完成: 10.2s
  Worker 0: build 7.0s, 139,255 neurons
  Worker 1: build 7.1s, 139,255 neurons


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:   10.1s finished
WARNING    The object 'neurons' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/Users/charlottel/MyLibrary/connectome/LIFmodel/LIF_simulation/flylif/core/network.py', line 295, in build_network
    neu = NeuronGroup( [brian2.core.base.unused_brian_object]
WARNING    The object 'synapses' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/Users/charlottel/MyLibrary/connectome/LIFmodel/LIF_simulation/flylif/core/network.py', line 317, in build_network
    syn = Synapses(neu, neu, 'w : volt', on_pre='g += w', [

In [81]:
# ==================== Cell 8.3_debug2: 测试单个control仿真 ====================

from joblib import Parallel, delayed

def test_control_worker(freq, neu_exc, data, params, target_neurons, n_trials):
    """测试完整的control仿真（不含silencing）"""
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    from brian2 import Hz, ms
    import gc
    
    print(f"  Worker: freq={freq} started")
    
    # Build
    columns = data['columns']
    NET = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    print(f"  Worker: freq={freq} network built, running simulation...")
    
    # Run simulation
    result = run_simulation(
        net_components=NET,
        neu_exc=neu_exc,
        params={'r_poi': freq * Hz},
        n_trials=n_trials,
        verbose=False
    )
    
    df = result['df']
    duration_s = float(params['t_run'] / ms) / 1000
    
    # Calculate MN9
    mn9_rates = []
    for trial in range(n_trials):
        trial_df = df[df['trial'] == trial]
        count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                   for tid in target_neurons)
        mn9_rates.append(count / duration_s)
    
    stats = {
        'mean': np.mean(mn9_rates),
        'std': np.std(mn9_rates),
    }
    
    print(f"  Worker: freq={freq} complete, MN9={stats['mean']:.1f} Hz")
    
    del NET
    gc.collect()
    
    return (freq, stats)

# Test: 只跑2个control（最简单）
print("\n测试：2个control仿真（50Hz + 100Hz）...")
print("⏱️  预计：2 × 30秒 ≈ 1分钟\n")

t0 = time()

tasks = [
    (50, NEU_ACTIVATE, DATA, DEFAULT_PARAMS, TARGET_MN9_TEST, N_TRIALS_TEST1),
    (100, NEU_ACTIVATE, DATA, DEFAULT_PARAMS, TARGET_MN9_TEST, N_TRIALS_TEST1),
]

results = Parallel(n_jobs=2, verbose=10)(
    delayed(test_control_worker)(*task) for task in tasks
)

print(f"\n✅ 完成: {time()-t0:.1f}s")

for freq, stats in results:
    print(f"  {freq} Hz: MN9 = {stats['mean']:.1f} ± {stats['std']:.1f} Hz")

print_memory("\nFinal memory: ")


测试：2个control仿真（50Hz + 100Hz）...
⏱️  预计：2 × 30秒 ≈ 1分钟



[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


  Worker: freq=100 started
  Worker: freq=100 network built, running simulation...

✅ 完成: 154.0s
  50 Hz: MN9 = 25.0 ± 6.5 Hz
  100 Hz: MN9 = 85.7 ± 7.2 Hz

Final memory: Memory: 60.1% used, 6.9GB free, Swap: 0.0GB


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:  2.6min finished


{'used_gb': 7.849664512,
 'available_gb': 6.853738496,
 'percent': 60.1,
 'swap_gb': 0.022806528}

  Worker: freq=100 complete, MN9=85.7 Hz
  Worker: freq=50 started
  Worker: freq=50 network built, running simulation...
  Worker: freq=50 complete, MN9=25.0 Hz


### test

In [82]:
# ==================== Cell 8.3C: Exp3测试1完整版（6 tasks） ====================

print("\n" + "=" * 70)
print("Exp3 Test 1 Complete: Controls + Silencing")
print("=" * 70)

# Create all tasks
tasks_complete = []

# Control tasks (reuse from debug2 if you want, or rebuild)
print("Creating tasks...")
for freq in FREQS_TEST1:
    tasks_complete.append(('control', None, freq, NEU_ACTIVATE, DATA,
                          DEFAULT_PARAMS, TARGET_MN9_TEST, N_TRIALS_TEST1))

# Silencing tasks
for freq in FREQS_TEST1:
    for neuron_id in NEURONS_SILENCE_TEST:
        tasks_complete.append(('silencing', neuron_id, freq, NEU_ACTIVATE, DATA,
                              DEFAULT_PARAMS, TARGET_MN9_TEST, N_TRIALS_TEST1))

print(f"\nTotal tasks: {len(tasks_complete)}")
print(f"  Controls: {len(FREQS_TEST1)}")
print(f"  Silencing: {len(NEURONS_SILENCE_TEST) * len(FREQS_TEST1)}")
print(f"Workers: 2")
print(f"\n⏱️  Expected: ~4 minutes (6 tasks × 77s / 2 workers)")

t0_complete = time()

with MemoryMonitor(label="Exp3-Test1-Complete"):
    results_complete = Parallel(n_jobs=2, verbose=10)(
        delayed(exp3_single_condition_worker)(*task) for task in tasks_complete
    )

complete_time = time() - t0_complete

print(f"\n✅ Test 1 complete: {complete_time/60:.1f} min")
print_memory("\nFinal memory: ")

# Organize results
results_test1_full = {}
for condition_type, freq, neuron_id, stats in results_complete:
    if freq not in results_test1_full:
        results_test1_full[freq] = {'control': None, 'silencing': {}}
    
    if condition_type == 'control':
        results_test1_full[freq]['control'] = stats
    else:
        results_test1_full[freq]['silencing'][neuron_id] = stats

# Calculate relative %
for freq in FREQS_TEST1:
    ctrl_mean = results_test1_full[freq]['control']['mean']
    for nid, stats in results_test1_full[freq]['silencing'].items():
        stats['relative_%'] = (stats['mean'] / ctrl_mean * 100) if ctrl_mean > 0 else 100

# Display
print("\n" + "=" * 70)
print("Exp3 Test 1 Results")
print("=" * 70)

for freq in FREQS_TEST1:
    ctrl = results_test1_full[freq]['control']
    silencing = results_test1_full[freq]['silencing']
    
    print(f"\n{freq} Hz:")
    print(f"  Control: {ctrl['mean']:.1f} ± {ctrl['std']:.1f} Hz")
    print(f"  Silencing results:")
    
    for nid, stats in silencing.items():
        required = "✓ Required" if stats['relative_%'] < 80 else ""
        print(f"    {nid}: {stats['mean']:.1f} Hz ({stats['relative_%']:.1f}%) {required}")

print("\n✅ Test 1 validation complete")
print(f"   Total time: {complete_time/60:.1f} min")
print(f"   Next: If successful → Test 2 (more neurons)")


Exp3 Test 1 Complete: Controls + Silencing
Creating tasks...

Total tasks: 6
  Controls: 2
  Silencing: 4
Workers: 2

⏱️  Expected: ~4 minutes (6 tasks × 77s / 2 workers)
[Exp3-Test1-Complete] Start - Memory: 58.5%


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:  1.4min
[Parallel(n_jobs=2)]: Done   4 out of   6 | elapsed:  2.8min remaining:  1.4min


[Exp3-Test1-Complete] End - Memory: 59.6% (Δ+0.2GB)

✅ Test 1 complete: 4.2 min

Final memory: Memory: 59.6% used, 6.9GB free, Swap: 0.0GB

Exp3 Test 1 Results

50 Hz:
  Control: 19.4 ± 8.4 Hz
  Silencing results:
    720575940624963786: 18.1 Hz (93.3%) 
    720575940622695448: 37.0 Hz (190.7%) 

100 Hz:
  Control: 90.5 ± 4.4 Hz
  Silencing results:
    720575940624963786: 83.6 Hz (92.4%) 
    720575940622695448: 91.4 Hz (101.0%) 

✅ Test 1 validation complete
   Total time: 4.2 min
   Next: If successful → Test 2 (more neurons)


[Parallel(n_jobs=2)]: Done   6 out of   6 | elapsed:  4.2min finished


# Exp3 Test 2: Single Neuron × 8 Frequencies (Internal Batching)

## Purpose
Test if a single neuron can handle all 8 frequencies with 1 network build.
Validates memory accumulation over 80 simulations (8 freqs × 10 trials).

## Configuration
- **Neuron**: 1 neuron (ID 21, first non-GRN)
- **Frequencies**: 8 frequencies [50, 60, 70, 80, 90, 100, 110, 120] Hz
- **Internal batching**: Split into 2 sub-batches (4 freqs each)
  - Batch 1: [50, 60, 70, 80] → 40 simulations
  - Batch 2: [90, 100, 110, 120] → 40 simulations
- **Trials**: 10/frequency
- **Workers**: 1 (serial for clean observation)

## Expected Runtime
- Build: 7 seconds × 1 = 7s
- Simulations: 8 × 25s = 200s
- **Total**: ~3.5 minutes

## Memory Watch Points
1. After build
2. After internal batch 1 (40 sims)
3. After internal batch 2 (80 sims total)
4. After cleanup

Expected accumulation: <500MB (acceptable)

---

In [22]:
# ==================== Cell 8.4B: 定义内部分批Worker ====================

def exp3_neuron_batched_worker(neuron_id, all_freqs, neu_exc, data, params, 
                               target_neurons, n_trials):
    """
    Process single neuron across all frequencies with internal batching.
    
    Builds network once, processes frequencies in 2 internal batches.
    
    Parameters
    ----------
    neuron_id : int
        Neuron to silence
    all_freqs : list
        All frequencies to test (e.g., [50, 60, ..., 120])
    neu_exc : list
        Neurons to activate (Sugar GRNs)
    data : dict
        Preprocessed data
    params : dict
        Parameters
    target_neurons : list
        Target neurons (MN9)
    n_trials : int
        Trials per frequency
    
    Returns
    -------
    tuple : (neuron_id, {freq: {'mean': float, 'std': float, 'relative_%': float}})
    """
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    from brian2 import Hz, ms
    import gc
    import numpy as np
    
    print(f"  [Neuron {neuron_id}] Worker started")
    from time import time
    t0_neuron = time()
    
    # Build network once
    columns = data['columns']
    NET = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=True
    )
    
    print(f"  [Neuron {neuron_id}] Network built")
    
    duration_s = float(params['t_run'] / ms) / 1000
    results = {}
    
    # Split frequencies into 2 internal batches
    mid_point = len(all_freqs) // 2
    freq_batch1 = all_freqs[:mid_point]
    freq_batch2 = all_freqs[mid_point:]
    
    # === Internal Batch 1 ===
    print(f"  [Neuron {neuron_id}] Internal batch 1: {freq_batch1}")
    
    for freq in freq_batch1:
        result = run_simulation(
            net_components=NET,
            neu_exc=neu_exc,
            neu_slnc=[neuron_id],
            params={'r_poi': freq * Hz},
            n_trials=n_trials,
            verbose=False
        )
        
        df = result['df']
        
        # Calculate target rates
        target_rates = []
        for trial in range(n_trials):
            trial_df = df[df['trial'] == trial]
            count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                       for tid in target_neurons)
            target_rates.append(count / duration_s)
        
        results[freq] = {
            'mean': np.mean(target_rates),
            'std': np.std(target_rates),
        }
    
    # Cleanup after batch 1
    gc.collect()
    print(f"  [Neuron {neuron_id}] Batch 1 done, gc.collect() called")
    
    # === Internal Batch 2 ===
    print(f"  [Neuron {neuron_id}] Internal batch 2: {freq_batch2}")
    
    for freq in freq_batch2:
        result = run_simulation(
            net_components=NET,
            neu_exc=neu_exc,
            neu_slnc=[neuron_id],
            params={'r_poi': freq * Hz},
            n_trials=n_trials,
            verbose=False
        )
        
        df = result['df']
        
        target_rates = []
        for trial in range(n_trials):
            trial_df = df[df['trial'] == trial]
            count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                       for tid in target_neurons)
            target_rates.append(count / duration_s)
        
        results[freq] = {
            'mean': np.mean(target_rates),
            'std': np.std(target_rates),
        }
    
    # Final cleanup
    del NET
    gc.collect()
    
    elapsed = time() - t0_neuron
    print(f"  [Neuron {neuron_id}] Complete: {elapsed:.1f}s, {len(results)} freqs")
    
    return (neuron_id, results)

print("✅ Batched worker function defined")

✅ Batched worker function defined


In [92]:
# ==================== Cell 8.5_debug: 测试worker中单次仿真 ====================

def exp3_single_sim_test(neuron_id, freq, neu_exc, data, params, target_neurons, n_trials):
    """Worker中只调用1次run_simulation"""
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    from brian2 import Hz, ms
    import gc
    
    print(f"  Worker started: neuron={neuron_id}, freq={freq}")
    
    # Build
    from time import time
    t0 = time()
    columns = data['columns']
    NET = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    print(f"  Network built ({time()-t0:.1f}s), starting simulation...")
    t1 = time()
    
    # === 只调用1次run_simulation ===
    result = run_simulation(
        net_components=NET,
        neu_exc=neu_exc,
        neu_slnc=[neuron_id],
        params={'r_poi': freq * Hz},
        n_trials=n_trials,
        verbose=True  # ← 打开verbose看内部发生什么
    )
    
    sim_time = time() - t1
    print(f"  Simulation done ({sim_time:.1f}s)")
    
    df = result['df']
    duration_s = float(params['t_run'] / ms) / 1000
    
    # Calculate
    target_rates = []
    for trial in range(n_trials):
        trial_df = df[df['trial'] == trial]
        count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                   for tid in target_neurons)
        target_rates.append(count / duration_s)
    
    stats = {'mean': np.mean(target_rates), 'std': np.std(target_rates)}
    
    print(f"  MN9: {stats['mean']:.1f} Hz")
    
    del NET
    gc.collect()
    
    return (neuron_id, freq, stats)

# 测试：只1个task
print("测试：Worker中调用1次run_simulation（verbose=True）")
print("观察：是否能完成，还是卡住\n")

task = (top_200_neurons[21], 50, NEU_ACTIVATE, DATA,
        DEFAULT_PARAMS, TARGET_MN9_TEST, 10)

t0 = time()

result = Parallel(n_jobs=1, verbose=10)(
    [delayed(exp3_single_sim_test)(*task)]
)

print(f"\n✅ 完成: {time()-t0:.1f}s")
print(result)

测试：Worker中调用1次run_simulation（verbose=True）
观察：是否能完成，还是卡住

  Worker started: neuron=720575940622695448, freq=50
  Network built (6.2s), starting simulation...

>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 1
    Duration: 1. s, Trials: 10
    Frequency: 50. Hz
    Silenced synapses: 109
    🚀 Running...
    Trial 10/10 (74.0s)
    ⏱️  Total: 707.7s (70.8s/trial)
    📊 Spikes: 35,947, Active neurons: 272
  Simulation done (726.8s)
  MN9: 39.7 Hz

✅ 完成: 750.1s
[(720575940622695448, 50, {'mean': np.float64(39.7), 'std': np.float64(8.54458893101359)})]


[Parallel(n_jobs=1)]: Done 1 out of 1 | elapsed: 12.5min
[Parallel(n_jobs=1)]: Done 1 out of 1 | elapsed: 12.5min finished


---

 🎯 **这个测试验证什么**

**关键问题**：
- ✅ 同一网络上80次仿真会累积多少内存？
- ✅ 内部分2批（40+40）的gc.collect()有效吗？

**成功标准**：
- ✅ 完成3.5分钟内
- ✅ 内存增长<500MB
- ✅ Worker verbose输出看到"Batch 1 done"和"Batch 2 done"

**如果成功** → 扩展到**20 neurons并行**：
```
20 neurons × (8 freqs × 10T) = 20 tasks
每个task内部处理80次仿真
20 / 4 workers = 5 tasks/worker ← 安全
总时间：5 × 3.5分钟 ≈ 17分钟（vs 160 tasks的40分钟）

# Exp3 Test 3: Single Neuron × 8 Frequencies (Network Reuse)

## Purpose
Validate network reuse strategy: 1 build, 8 frequencies in 2 internal batches.

## Configuration
- **Neuron**: 1 neuron (first non-GRN from top_200)
- **Frequencies**: 8 frequencies [50, 60, 70, 80, 90, 100, 110, 120] Hz
- **Trials**: 10/frequency
- **Internal batching**: 
  - Batch 1: [50, 60, 70, 80] → 40 simulations
  - Batch 2: [90, 100, 110, 120] → 40 simulations
- **Workers**: 1 (serial)
- **Network builds**: 1 (shared across all frequencies)

## Critical Metrics
- **Time**: ~8 minutes expected
- **Memory accumulation**: 
  - After batch 1 (40 sims): Check increase
  - After batch 2 (80 sims): Check total increase
  - **Success criteria**: < 1GB increase

## Why This Matters
If successful → Can extend to 20 neurons parallel for full experiment.
If memory grows >1GB → Need smaller batches or different strategy.



In [20]:
# ==================== Cell 8.5B: 运行Exp3测试3（网络复用） ====================

from joblib import Parallel, delayed

print("=" * 70)
print("Exp3 Test 3: Single Neuron × 8 Freqs (Network Reuse)")
print("=" * 70)

# === 找到第一个非GRN神经元 ===
first_non_grn = None
for i, nid in enumerate(top_200_neurons):
    if nid not in NEU_SUGAR_LEFT:
        first_non_grn = nid
        first_non_grn_idx = i
        print(f"\nFirst non-GRN neuron:")
        print(f"  Position: {i}")
        print(f"  ID: {nid}")
        break

if first_non_grn is None:
    raise ValueError("No non-GRN found in top 200!")

# Configuration
NEURON_TEST3 = first_non_grn
# FREQS_TEST3_FULL = [50, 60, 70, 80, 90, 100, 110, 120]  # 8 frequencies
FREQS_TEST3_FULL = [50, 60, 70, 80]  # 2 frequencies
N_TRIALS_TEST3 = 10

print(f"\nConfiguration:")
print(f"  Neuron to silence: {NEURON_TEST3}")
print(f"  Activated: {len(NEU_ACTIVATE)} Sugar GRNs (continuous)")
print(f"  Frequencies: {FREQS_TEST3_FULL}")
print(f"  Trials/freq: {N_TRIALS_TEST3}")
print(f"  Total simulations: {len(FREQS_TEST3_FULL) * N_TRIALS_TEST3} (on 1 network)")
print(f"  Internal batching: 4 + 4 freqs")
print(f"  Workers: 1")
print(f"\n⏱️  Expected: ~8 minutes (1 build + 8×60s simulations)")

Exp3 Test 3: Single Neuron × 8 Freqs (Network Reuse)

First non-GRN neuron:
  Position: 21
  ID: 720575940622695448

Configuration:
  Neuron to silence: 720575940622695448
  Activated: 21 Sugar GRNs (continuous)
  Frequencies: [50, 60, 70, 80]
  Trials/freq: 10
  Total simulations: 40 (on 1 network)
  Internal batching: 4 + 4 freqs
  Workers: 1

⏱️  Expected: ~8 minutes (1 build + 8×60s simulations)


In [ ]:
print_memory("\nBefore test: ")

# === 运行（使用已定义的exp3_neuron_batched_worker）===
# 注意：这个函数在Cell 8.4B已经定义

task = (NEURON_TEST3, FREQS_TEST3_FULL, NEU_ACTIVATE, DATA,
        DEFAULT_PARAMS, TARGET_MN9_TEST, N_TRIALS_TEST3)

print("\nStarting test...")
print("Watch for these checkpoints in output:")
print("  1. 'Network built'")
print("  2. 'Internal batch 1: [50, 60, 70, 80]'")
print("  3. 'Batch 1 done, gc.collect() called'")
print("  4. 'Internal batch 2: [90, 100, 110, 120]'")
print("  5. 'Complete: XXs'")
print()

t0_test3 = time()

with MemoryMonitor(label="Exp3-Test3"):
    result_test3 = Parallel(n_jobs=1, verbose=10)(
        [delayed(exp3_neuron_batched_worker)(*task)]
    )

test3_time = time() - t0_test3

print(f"\n✅ Test 3 complete: {test3_time/60:.1f} min")
print_memory("\nFinal memory: ")

# === 结果展示 ===
neuron_id, freq_results = result_test3[0]

print("\n" + "=" * 70)
print(f"Results: Neuron {neuron_id}")
print("=" * 70)
print(f"{'Freq (Hz)':<10} {'MN9 Mean (Hz)':<15} {'MN9 Std (Hz)':<15}")
print("-" * 40)

for freq in sorted(freq_results.keys()):
    stats = freq_results[freq]
    print(f"{freq:<10} {stats['mean']:<15.1f} {stats['std']:<15.1f}")

# === 内存累积分析 ===
print("\n" + "=" * 70)
print("Memory Accumulation Analysis")
print("=" * 70)
print("Check MemoryMonitor output above:")
print("  - Start memory: XX%")
print("  - End memory: YY%")
print(f"  - Increase: (YY - XX)% → Expected < 6% (~1GB)")
print()
print("If memory increase < 1GB:")
print("  ✅ Can safely extend to 5 neurons parallel")
print("  ✅ Cloud strategy validated")
print()
print("If memory increase > 1GB:")
print("  ⚠️  Need more aggressive batching")
print("  ⚠️  Or reduce to 4 freqs per internal batch")

### debug

In [24]:
# ==================== Cell 8.6A_debug: 主进程循环测试（排除joblib因素） ====================

print("=" * 70)
print("Debug A: 主进程中连续2次run_simulation")
print("=" * 70)
print("目的：排除joblib/worker的影响，测试run_simulation本身的循环问题")
print()

# 使用已存在的DATA构建网络（主进程中）
from flylif.core.network import build_network
from flylif.core.simulation import run_simulation
from brian2 import Hz, ms

# 选择测试神经元
test_neuron = top_200_neurons[21]  # 非GRN
print(f"Test neuron: {test_neuron}")
print(f"Activated: {len(NEU_ACTIVATE)} Sugar GRNs")

# Build network
print("\nBuilding network in main process...")
t0 = time()

columns = DATA['columns']
NET_main = build_network(
    data=DATA,
    pre_col=columns['pre_col'],
    post_col=columns['post_col'],
    weight_col=columns['weight_col'],
    nt_prob_cols=columns.get('nt_prob_cols', {}),
    params=DEFAULT_PARAMS,
    syn_threshold=5,
    verbose=False
)

print(f"✅ Built in {time()-t0:.1f}s")
print_memory("After build: ")

# === 第1次run_simulation ===
print("\n" + "=" * 70)
print("第1次run_simulation @ 50 Hz")
print("=" * 70)

t0_sim1 = time()

result1 = run_simulation(
    net_components=NET_main,
    neu_exc=NEU_ACTIVATE,
    neu_slnc=[test_neuron],
    params={'r_poi': 50 * Hz},
    n_trials=10,
    verbose=True  # ← 打开verbose
)

sim1_time = time() - t0_sim1

print(f"\n✅ 第1次完成: {sim1_time:.1f}s")
print(f"   Spikes: {result1['n_spikes']:,}")
print(f"   Active neurons: {result1['n_active']:,}")

print_memory("After 1st sim: ")

# === 第2次run_simulation ===
print("\n" + "=" * 70)
print("第2次run_simulation @ 60 Hz")
print("=" * 70)
print("⚠️  Watch: Does it start? Does it complete?")
print()

t0_sim2 = time()

result2 = run_simulation(
    net_components=NET_main,
    neu_exc=NEU_ACTIVATE,
    neu_slnc=[test_neuron],
    params={'r_poi': 60 * Hz},
    n_trials=10,
    verbose=True  # ← 打开verbose
)

sim2_time = time() - t0_sim2

print(f"\n✅ 第2次完成: {sim2_time:.1f}s")
print(f"   Spikes: {result2['n_spikes']:,}")

print_memory("After 2nd sim: ")

# Cleanup
del NET_main
gc.collect()

print("\n" + "=" * 70)
print("✅ 主进程循环测试完成")
print("=" * 70)
print(f"结论：主进程中连续调用run_simulation {'成功' if sim2_time < 1000 else '异常'}")

Debug A: 主进程中连续2次run_simulation
目的：排除joblib/worker的影响，测试run_simulation本身的循环问题

Test neuron: 720575940622695448
Activated: 21 Sugar GRNs

Building network in main process...
✅ Built in 6.0s
After build: Memory: 55.5% used, 7.6GB free, Swap: 0.0GB

第1次run_simulation @ 50 Hz

>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 1
    Duration: 1. s, Trials: 10
    Frequency: 50. Hz
    Silenced synapses: 109
    🚀 Running...
    Trial 10/10 (59.6s)
    ⏱️  Total: 566.0s (56.6s/trial)
    📊 Spikes: 35,452, Active neurons: 266

✅ 第1次完成: 575.9s
   Spikes: 35,452
   Active neurons: 266
After 1st sim: Memory: 56.2% used, 7.5GB free, Swap: 0.0GB

第2次run_simulation @ 60 Hz
⚠️  Watch: Does it start? Does it complete?


>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 1
    Duration: 1. s, Trials: 10
    Frequency: 60. Hz
    Silenced synapses: 109
    🚀 Running...
    Trial 10/10 (68.3s)
    ⏱️  Total: 630.2s (63.0s/trial)
    📊 Spikes: 51,848, Active neu

In [26]:
# ==================== Cell 8.6B_debug: Worker循环测试（verbose+flush） ====================

from joblib import Parallel, delayed
import sys

def exp3_worker_verbose_flush(neuron_id, freqs, neu_exc, data, params, 
                               target_neurons, n_trials):
    """
    Worker测试：打开verbose并强制flush输出。
    """
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    from brian2 import Hz, ms
    import gc
    from time import time
    
    # 强制刷新输出
    print(f"[Worker PID={os.getpid()}] Started", flush=True)
    
    # Build
    t0 = time()
    columns = data['columns']
    NET = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    print(f"[Worker] Built in {time()-t0:.1f}s", flush=True)
    
    results = {}
    duration_s = float(params['t_run'] / ms) / 1000
    
    # === 循环2次，强制flush ===
    for i, freq in enumerate(freqs, 1):
        print(f"\n[Worker] ========== Iteration {i}/{len(freqs)}: {freq} Hz ==========", flush=True)
        t0_sim = time()
        
        # 打开verbose观察内部
        result = run_simulation(
            net_components=NET,
            neu_exc=neu_exc,
            neu_slnc=[neuron_id],
            params={'r_poi': freq * Hz},
            n_trials=n_trials,
            verbose=True  # ← Verbose
        )
        
        sim_time = time() - t0_sim
        print(f"[Worker] Iteration {i} COMPLETE ({sim_time:.1f}s)", flush=True)
        
        df = result['df']
        
        # Calculate
        target_rates = []
        for trial in range(n_trials):
            trial_df = df[df['trial'] == trial]
            count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                       for tid in target_neurons)
            target_rates.append(count / duration_s)
        
        results[freq] = {
            'mean': np.mean(target_rates),
            'std': np.std(target_rates)
        }
        
        print(f"[Worker] MN9: {results[freq]['mean']:.1f} Hz", flush=True)
        gc.collect()
    
    print(f"\n[Worker] ALL DONE: {len(results)} freqs", flush=True)
    
    del NET
    gc.collect()
    
    return (neuron_id, results)

In [28]:
# === 运行 ===
import os

print("=" * 70)
print("Debug B: Worker中循环2次（verbose + flush）")
print("=" * 70)
print(f"Main process PID: {os.getpid()}")

test_neuron = top_200_neurons[21]
test_freqs = [50, 60]

print(f"\nConfiguration:")
print(f"  Neuron: {test_neuron}")
print(f"  Frequencies: {test_freqs}")
print(f"  Trials: 10")
print(f"  Workers: 1")
print(f"\n⏱️  Expected: ~20 minutes (based on main process test)")
print(f"\n⚠️  Watch for verbose output from worker process\n")

print_memory("Before: ")

t0 = time()

task = (test_neuron, test_freqs, NEU_ACTIVATE, DATA,
        DEFAULT_PARAMS, TARGET_MN9_TEST, 4)

result = Parallel(n_jobs=1, verbose=10)(
    [delayed(exp3_worker_verbose_flush)(*task)]
)

total_time = time() - t0

print(f"\n✅ Worker test complete: {total_time/60:.1f} min")
print_memory("After: ")

neuron_id, freq_results = result[0]
print(f"\nResults: {freq_results}")

print("\n" + "=" * 70)
print("对比：")
print(f"  主进程循环: 1220秒 ✅")
print(f"  Worker循环: {total_time:.0f}秒 {'✅' if total_time < 1500 else '❌'}")
print("=" * 70)

Debug B: Worker中循环2次（verbose + flush）
Main process PID: 19630

Configuration:
  Neuron: 720575940622695448
  Frequencies: [50, 60]
  Trials: 10
  Workers: 1

⏱️  Expected: ~20 minutes (based on main process test)

⚠️  Watch for verbose output from worker process

Before: Memory: 62.7% used, 6.4GB free, Swap: 0.0GB
[Worker PID=19630] Started
[Worker] Built in 6.1s

[Worker] ========== Iteration 1/2: 50 Hz ==========

>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 1
    Duration: 1. s, Trials: 4
    Frequency: 50. Hz
    Silenced synapses: 109
    🚀 Running...
    Trial 4/4 (66.2s)
    ⏱️  Total: 263.2s (65.8s/trial)
    📊 Spikes: 14,109, Active neurons: 237
[Worker] Iteration 1 COMPLETE (278.8s)
[Worker] MN9: 38.5 Hz

[Worker] ========== Iteration 2/2: 60 Hz ==========

>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 1
    Duration: 1. s, Trials: 4
    Frequency: 60. Hz
    Silenced synapses: 109
    🚀 Running...
    Trial 4/4 (69.3s)
   

[Parallel(n_jobs=1)]: Done 1 out of 1 | elapsed: 10.4min
[Parallel(n_jobs=1)]: Done 1 out of 1 | elapsed: 10.4min finished


# Exp3 Test 3: Single Neuron × 8 Frequencies (Full Reuse Strategy)

**Purpose**
Validate network reuse across all 8 frequencies with internal batching.
This is the critical test for cloud deployment strategy.

**Configuration**
- **Neuron**: 1 neuron (first non-GRN)
- **Frequencies**: 8 frequencies [50, 60, 70, 80, 90, 100, 110, 120] Hz
- **Trials**: 4/frequency (reduced for faster validation)
- **Internal batching**: 
  - Batch 1: [50, 60, 70, 80] → 4 freqs × 4T = 16 sims
  - Batch 2: [90, 100, 110, 120] → 4 freqs × 4T = 16 sims
- **Workers**: 1 (serial)
- **Network builds**: 1 (shared across all)

**Expected Performance**
- Build: 7 seconds
- Simulations: 8 × (60s × 4T / 10T) ≈ 192 seconds
- **Total**: ~3.5 minutes with 4 trials

**Success Criteria**
- ✅ Completes without hanging
- ✅ Memory increase < 1GB
- ✅ All 8 frequencies return valid results

**Why This Matters**
If successful → 20 neurons can be processed in parallel safely.
Cloud deployment: 200 neurons / 48 workers = 4 neurons/worker (safe).

---

In [29]:
# ==================== Cell 8.7B: 运行Exp3测试3（8频率×4T） ====================

from joblib import Parallel, delayed
import os
import sys

def exp3_test3_worker(neuron_id, all_freqs, neu_exc, data, params, 
                      target_neurons, n_trials):
    """
    Single neuron, all 8 frequencies, with internal batching.
    Based on successful verbose test (Test B).
    """
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    from brian2 import Hz, ms
    import gc
    import numpy as np
    from time import time
    
    print(f"[Worker PID={os.getpid()}] Started: neuron={neuron_id}", flush=True)
    
    # Build network once
    t0_build = time()
    columns = data['columns']
    NET = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    build_time = time() - t0_build
    print(f"[Worker] Network built ({build_time:.1f}s)", flush=True)
    
    duration_s = float(params['t_run'] / ms) / 1000
    results = {}
    
    # Split into 2 internal batches
    mid = len(all_freqs) // 2
    freq_batch1 = all_freqs[:mid]
    freq_batch2 = all_freqs[mid:]
    
    # === Internal Batch 1 ===
    print(f"[Worker] Internal batch 1: {freq_batch1}", flush=True)
    
    for i, freq in enumerate(freq_batch1, 1):
        print(f"[Worker]   → Freq {i}/{len(freq_batch1)}: {freq} Hz...", flush=True)
        t0_sim = time()
        
        result = run_simulation(
            net_components=NET,
            neu_exc=neu_exc,
            neu_slnc=[neuron_id],
            params={'r_poi': freq * Hz},
            n_trials=n_trials,
            verbose=True  # Verbose to see trial progress
        )
        
        sim_time = time() - t0_sim
        print(f"[Worker]   ✓ Done ({sim_time:.1f}s)", flush=True)
        
        df = result['df']
        
        # Calculate stats
        target_rates = []
        for trial in range(n_trials):
            trial_df = df[df['trial'] == trial]
            count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                       for tid in target_neurons)
            target_rates.append(count / duration_s)
        
        results[freq] = {
            'mean': np.mean(target_rates),
            'std': np.std(target_rates)
        }
        
        print(f"[Worker]   MN9: {results[freq]['mean']:.1f} Hz", flush=True)
    
    # Cleanup after batch 1
    gc.collect()
    print(f"[Worker] Batch 1 complete, gc.collect() called", flush=True)
    
    # === Internal Batch 2 ===
    print(f"\n[Worker] Internal batch 2: {freq_batch2}", flush=True)
    
    for i, freq in enumerate(freq_batch2, 1):
        print(f"[Worker]   → Freq {i}/{len(freq_batch2)}: {freq} Hz...", flush=True)
        t0_sim = time()
        
        result = run_simulation(
            net_components=NET,
            neu_exc=neu_exc,
            neu_slnc=[neuron_id],
            params={'r_poi': freq * Hz},
            n_trials=n_trials,
            verbose=True
        )
        
        sim_time = time() - t0_sim
        print(f"[Worker]   ✓ Done ({sim_time:.1f}s)", flush=True)
        
        df = result['df']
        
        target_rates = []
        for trial in range(n_trials):
            trial_df = df[df['trial'] == trial]
            count = sum(len(trial_df[trial_df['flywire_id'] == tid])
                       for tid in target_neurons)
            target_rates.append(count / duration_s)
        
        results[freq] = {
            'mean': np.mean(target_rates),
            'std': np.std(target_rates)
        }
        
        print(f"[Worker]   MN9: {results[freq]['mean']:.1f} Hz", flush=True)
    
    # Final cleanup
    del NET
    gc.collect()
    
    total_time = time() - t0_build
    print(f"\n[Worker] COMPLETE: {total_time:.1f}s ({len(results)} freqs)", flush=True)
    
    return (neuron_id, results)

In [30]:
# === 运行测试 ===
print("=" * 70)
print("Exp3 Test 3: Single Neuron × 8 Frequencies")
print("=" * 70)

# 找到非GRN神经元
test_neuron_t3 = None
for i, nid in enumerate(top_200_neurons):
    if nid not in NEU_SUGAR_LEFT:
        test_neuron_t3 = nid
        print(f"\nTest neuron (Position {i}): {nid}")
        break

FREQS_TEST3_FULL = [50, 60, 70, 80, 90, 100, 110, 120]
N_TRIALS_TEST3 = 4  # Start with 4 trials

print(f"\nConfiguration:")
print(f"  Neuron: {test_neuron_t3}")
print(f"  Frequencies: {FREQS_TEST3_FULL}")
print(f"  Trials/freq: {N_TRIALS_TEST3}")
print(f"  Total simulations: {len(FREQS_TEST3_FULL) * N_TRIALS_TEST3} (on 1 network)")
print(f"  Internal batching: {len(FREQS_TEST3_FULL)//2} + {len(FREQS_TEST3_FULL)//2} freqs")
print(f"  Workers: 1")
print(f"\n⏱️  Expected: ~9 minutes (7s build + 8×25s×4T)")

print_memory("\nBefore test: ")

t0_test3 = time()

task = (test_neuron_t3, FREQS_TEST3_FULL, NEU_ACTIVATE, DATA,
        DEFAULT_PARAMS, TARGET_MN9_TEST, N_TRIALS_TEST3)

print("\nStarting test (watch for verbose output from worker)...\n")

with MemoryMonitor(label="Exp3-Test3"):
    result_test3 = Parallel(n_jobs=1, verbose=10)(
        [delayed(exp3_test3_worker)(*task)]
    )

test3_time = time() - t0_test3

print(f"\n✅ Test 3 complete: {test3_time/60:.1f} min")
print_memory("\nFinal memory: ")

Exp3 Test 3: Single Neuron × 8 Frequencies

Test neuron (Position 21): 720575940622695448

Configuration:
  Neuron: 720575940622695448
  Frequencies: [50, 60, 70, 80, 90, 100, 110, 120]
  Trials/freq: 4
  Total simulations: 32 (on 1 network)
  Internal batching: 4 + 4 freqs
  Workers: 1

⏱️  Expected: ~9 minutes (7s build + 8×25s×4T)

Before test: Memory: 71.2% used, 5.0GB free, Swap: 0.1GB

Starting test (watch for verbose output from worker)...

[Exp3-Test3] Start - Memory: 71.2%
[Worker PID=19630] Started: neuron=720575940622695448
[Worker] Network built (6.5s)
[Worker] Internal batch 1: [50, 60, 70, 80]
[Worker]   → Freq 1/4: 50 Hz...

>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 1
    Duration: 1. s, Trials: 4
    Frequency: 50. Hz
    Silenced synapses: 109
    🚀 Running...
    Trial 4/4 (70.3s)
    ⏱️  Total: 285.4s (71.4s/trial)
    📊 Spikes: 14,407, Active neurons: 264
[Worker]   ✓ Done (304.0s)
[Worker]   MN9: 38.5 Hz
[Worker]   → Freq 2/4: 60 Hz...

[Parallel(n_jobs=1)]: Done 0 out of 1 | elapsed: 24.7min remaining: 24.7min


[Exp3-Test3] End - Memory: 64.1% (Δ+1.1GB)


KeyboardInterrupt: 

In [ ]:
# === 结果分析 ===
neuron_id, freq_results = result_test3[0]

print("\n" + "=" * 70)
print(f"Results: Neuron {neuron_id} (8 Frequencies)")
print("=" * 70)
print(f"{'Freq (Hz)':<10} {'MN9 Mean':<12} {'MN9 Std':<12}")
print("-" * 34)

for freq in sorted(freq_results.keys()):
    stats = freq_results[freq]
    print(f"{freq:<10} {stats['mean']:<12.1f} {stats['std']:<12.1f}")

# === 性能分析 ===
print("\n" + "=" * 70)
print("Performance Analysis")
print("=" * 70)
print(f"Total time: {test3_time/60:.1f} min")
print(f"Time/freq: {test3_time/len(FREQS_TEST3_FULL):.1f}s")
print(f"Network builds: 1")
print(f"Simulations: {len(FREQS_TEST3_FULL)} (reusing network)")

# Memory
mem_info = print_memory("Final: ")
print(f"\n💡 Memory accumulation from MemoryMonitor output:")
print(f"   If Δ < 1GB → Can extend to 5 neurons parallel")
print(f"   If Δ > 1GB → Need smaller batches")

print(f"\n✅ Network reuse strategy validated for 8 frequencies")

In [ ]:
---

## 🎯 **观察要点**

运行时关注：

1. **进度输出**（应该连续看到）：
```
   [Worker] → Freq 1/4: 50 Hz...
       Trial 1/4 (25s)
       Trial 4/4 (25s)
   [Worker] ✓ Done (100s)
   [Worker] MN9: XX Hz
   ...重复8次
```

2. **时间分布**：
   - 每个频率：25s × 4T = 100秒（合理）
   - 8个频率：800秒 ≈ 13分钟

3. **MemoryMonitor**：
```
   [Exp3-Test3] Start - Memory: 63%
   [Exp3-Test3] End - Memory: 68%?
   Δ = 5% ≈ 800MB（接近临界）